# Jaxpy Numpyro test

最小 smoke test：检查 `align='sph'` 的 JAM 路径能不能进入 NumPyro/NUTS。

- 这里先用本地 `jampy` 做一次 `sph` forward smoke test。
- 真正的 NumPyro 采样使用的是本地 `jaxpy`，因为 plain `jampy` 不是 JAX-traceable。
- 配置故意压得很小：`8` 个 bins，`num_warmup=3`，`num_samples=3`，只回答“能不能跑通”。


In [ ]:
import os
import sys
import time
import pickle as pkl
import importlib
from pathlib import Path

os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["MPLBACKEND"] = "Agg"

root = Path("/mnt/d/lensing/jaxpy/jaxpy/jaxpy")
sys.path.insert(0, str(root))
sys.path.insert(0, "/mnt/d/lensing/Jampy/jampy-8.1.4")

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import numpy as np
import numpyro
numpyro.set_platform("cpu")

from jax.tree_util import Partial as partial
from numpyro import distributions as dist
from numpyro.infer import MCMC, NUTS, init_to_value

import param_util
from SLCOSMO import tool
from MGE_jax import MGE
from jam_axi_proj_jax import jam_axi_proj

import jampy.axi.jam_axi_proj as jam_base_mod
jam_base_mod = importlib.reload(jam_base_mod)
from jampy.axi.jam_axi_proj import jam_axi_proj as jam_base

print("JAX backend:", jax.default_backend())
print("NumPyro platform: cpu")
print("Note: NumPyro/NUTS uses jaxpy, not plain jampy.")

lens_light = pkl.load(open(root / "lens_light_jackpot.pkl", "rb"))[0]
surf_lum = jnp.asarray(lens_light["amp"])
sigma_lum = jnp.asarray(lens_light["sigma"])
_, qobs_lum = param_util.ellipticity2phi_q(lens_light["e1"], lens_light["e2"])
qobs_lum = jnp.asarray(qobs_lum)

coords = np.array([
    [-1.0, -1.0], [0.0, -1.0], [1.0, -1.0],
    [-1.0,  0.0], [1.0,  0.0],
    [-1.0,  1.0], [0.0,  1.0], [1.0,  1.0],
], dtype=float)
xbin = coords[:, 0]
ybin = coords[:, 1]

theta_E_true = 1.397
gamma_true = 2.027
q_mass_true = 0.946
beta_true = 0.0
z_lens = 0.222
z_source = 0.609
err_sigma = 15.0

cosmology = {
    "Omegam": 0.3,
    "Omegak": 0.0,
    "w0": -1.0,
    "wa": 0.0,
    "h0": 70.0,
}
distance = float(np.asarray(tool.dldsdls(z_lens, z_source, cosmology, n=20)[0]).ravel()[0])

mge_epl = MGE(
    tool.EPL_msunmpc,
    "thetaE",
    n_gauss=10,
    n_terms=28,
    sigma_start_mult=1 / 100,
    sigma_end_mult=50,
)
surf_pot_true, sigma_pot_true = mge_epl.decompose(
    thetaE=theta_E_true,
    gamma=gamma_true,
    zl=z_lens,
    zs=z_source,
    cosmology=cosmology,
)
qobs_pot_true = jnp.full_like(surf_pot_true, q_mass_true)
beta_vec_true = jnp.full_like(surf_lum, beta_true)

sigmapsf = jnp.array([0.6 / 2.355])
normpsf = jnp.array([1.0])
pixsize = 0.6

common_kwargs = dict(
    surf_lum=surf_lum,
    sigma_lum=sigma_lum,
    qobs_lum=qobs_lum,
    inc=90.0,
    mbh=0.0,
    distance=distance,
    xbin=xbin,
    ybin=ybin,
    sigmapsf=sigmapsf,
    normpsf=normpsf,
    pixsize=pixsize,
    logistic=False,
    step=0.05,
    moment="zz",
    align="sph",
    ml=None,
    nrad=8,
    nang=6,
    nlos=200,
    quiet=True,
)

base_kwargs = {}
for key, value in dict(common_kwargs, surf_pot=surf_pot_true, sigma_pot=sigma_pot_true, qobs_pot=qobs_pot_true, beta=beta_vec_true).items():
    if key in {"inc", "mbh", "distance", "pixsize", "step"}:
        base_kwargs[key] = float(value)
    elif key in {"logistic", "moment", "align", "ml", "quiet"}:
        base_kwargs[key] = value
    else:
        base_kwargs[key] = np.asarray(value)

base_model = jam_base(**base_kwargs, plot=False).model
base_model = np.asarray(base_model)
print("base jampy sph forward finite:", bool(np.isfinite(base_model).all()))
print("base jampy sph mean sigma_los:", float(base_model.mean()))

jam_obj = jam_axi_proj()
jam_eval = partial(jam_obj.get_kinematics, **common_kwargs)
sigma_true, *_ = jam_eval(
    surf_pot=surf_pot_true,
    sigma_pot=sigma_pot_true,
    qobs_pot=qobs_pot_true,
    beta=beta_vec_true,
)
sigma_true = jnp.asarray(sigma_true)
print("jaxpy sph forward finite:", bool(np.isfinite(np.asarray(sigma_true)).all()))
print("jaxpy sph mean sigma_los:", float(np.asarray(sigma_true).mean()))
print("mean |base - jax|:", float(np.mean(np.abs(base_model - np.asarray(sigma_true)))))

sigma_obs = sigma_true + err_sigma * jax.random.normal(jax.random.PRNGKey(0), sigma_true.shape)
print("number of bins:", sigma_true.size)
print("mock sigma_los mean:", float(np.asarray(sigma_obs).mean()))


In [ ]:
def lens_kinematics_model(sigma_obs):
    theta_E = numpyro.sample("theta_E", dist.Uniform(1.25, 1.55))
    gamma = numpyro.sample("gamma", dist.Uniform(1.90, 2.15))

    surf_pot, sigma_pot = mge_epl.decompose(
        thetaE=theta_E,
        gamma=gamma,
        zl=z_lens,
        zs=z_source,
        cosmology=cosmology,
    )
    qobs_pot = jnp.full_like(surf_pot, q_mass_true)
    beta_vec = jnp.full_like(surf_lum, beta_true)

    pred_model, *_ = jam_eval(
        surf_pot=surf_pot,
        sigma_pot=sigma_pot,
        qobs_pot=qobs_pot,
        beta=beta_vec,
    )

    sigma = err_sigma * jnp.ones_like(pred_model)
    with numpyro.plate("bins", pred_model.size):
        numpyro.sample("obs", dist.Normal(pred_model, sigma), obs=sigma_obs)

nuts = NUTS(
    lens_kinematics_model,
    init_strategy=init_to_value(values={"theta_E": theta_E_true, "gamma": gamma_true}),
    target_accept_prob=0.7,
)
mcmc = MCMC(
    nuts,
    num_warmup=3,
    num_samples=3,
    num_chains=1,
    chain_method="sequential",
    progress_bar=False,
)

t0 = time.perf_counter()
mcmc.run(jax.random.PRNGKey(1), sigma_obs=sigma_obs)
elapsed_s = time.perf_counter() - t0

samples = mcmc.get_samples()
extra_fields = mcmc.get_extra_fields()

print(f"MCMC elapsed: {elapsed_s:.3f} s")
print("sample keys:", list(samples.keys()))
print("divergences:", int(np.asarray(extra_fields["diverging"]).sum()))


In [ ]:
posterior_mean = {key: float(np.asarray(value).mean()) for key, value in samples.items()}
posterior_std = {key: float(np.asarray(value).std(ddof=0)) for key, value in samples.items()}

print("posterior mean:", posterior_mean)
print("posterior std:", posterior_std)
print("true values:", {"theta_E": theta_E_true, "gamma": gamma_true})
